In [ ]:
import os
# os.chdir('../')
# %pwd

'c:\\Users\\singh\\OneDrive\\Desktop\\DataScienceProject'

In [2]:
%pwd

'c:\\Users\\singh\\OneDrive\\Desktop\\DataScienceProject'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

In [4]:
from src.datascience.constants import *
from src.datascience.utils.main_utils import read_yaml, create_directories

class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH,
                 params_filepath=PARAMS_FILE_PATH,
                 schema_filepath=SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config=self.config.model_trainer
        params=self.params.ElasticNet
        schema=self.schema.TARGET_COLUMN
        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train_data_path=config.train_data_path,
            test_data_path=config.test_data_path,
            model_name=config.model_name,
            alpha=params.alpha,
            l1_ratio=params.l1_ratio,
            target_column=schema.name
        )
        return model_trainer_config

In [5]:
import pandas as pd
import os
from src.datascience import logger

from sklearn.linear_model import ElasticNet
import joblib

class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config
    
    def train(self):
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        train_x = train_data.drop(columns= [self.config.target_column])
        test_x = test_data.drop(columns= [self.config.target_column])
        train_y = train_data[self.config.target_column]
        test_y = test_data[self.config.target_column]

        model = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio, random_state=42)
        model.fit(train_x, train_y)

        joblib.dump(model, os.path.join(self.config.root_dir, self.config.model_name))

In [6]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config= model_trainer_config)
    model_trainer.train()
except Exception as e:
    logger.error(e)
    raise e

[ 2026-05-12 17:01:11,942 : INFO : main_utils : YAML file: config\config.yaml Loaded Successfully ]
[ 2026-05-12 17:01:11,953 : INFO : main_utils : YAML file: params.yaml Loaded Successfully ]
[ 2026-05-12 17:01:11,956 : INFO : main_utils : YAML file: schema.yaml Loaded Successfully ]
[ 2026-05-12 17:01:11,958 : INFO : main_utils : Create Directory at: artifacts ]
[ 2026-05-12 17:01:11,960 : INFO : main_utils : Create Directory at: artifacts/model_trainer ]
